# Q3: Jack's Car Rental – Policy Iteration

**Silver Lecture 3, pages 14–15**

This notebook:
1. Explains the MDP formulation
2. Shows the precomputed transition matrices
3. Runs policy iteration to convergence
4. Reproduces the policy heatmap and value surface from the slides

> ⚠️ **Note:** Policy iteration takes ~1-2 minutes due to the 21×21 state space.

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np

from src.q3_jacks_car.jacks_car import (
    _EXP_REW_1,
    _TRANS_1,
    _TRANS_2,
    GAMMA,
    LAMBDA_RENT_1,
    LAMBDA_RENT_2,
    LAMBDA_RETURN_1,
    LAMBDA_RETURN_2,
    MAX_CARS,
    MAX_MOVE,
    MOVE_COST,
    RENTAL_REWARD,
    plot_results,
    poisson_pmf,
    policy_iteration,
)

print('Imports done (transition matrices already precomputed)')

## Problem Parameters

In [ ]:
params = {
    'Max cars / location': MAX_CARS,
    'Max cars moved / night': MAX_MOVE,
    'Rental revenue ($)': RENTAL_REWARD,
    'Moving cost ($/car)': MOVE_COST,
    'Discount γ': GAMMA,
    'λ_rent₁': LAMBDA_RENT_1,
    'λ_rent₂': LAMBDA_RENT_2,
    'λ_return₁': LAMBDA_RETURN_1,
    'λ_return₂': LAMBDA_RETURN_2,
}
for k, v in params.items():
    print(f'  {k:30s}: {v}')

## Poisson Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, lam, title in [
    (axes[0], LAMBDA_RENT_1, f'Rentals Loc1: Poisson({LAMBDA_RENT_1})'),
    (axes[1], LAMBDA_RENT_2, f'Rentals Loc2: Poisson({LAMBDA_RENT_2})'),
]:
    ks = np.arange(12)
    probs = [poisson_pmf(lam, k) for k in ks]
    ax.bar(ks, probs, color='steelblue', alpha=0.8, edgecolor='navy')
    ax.set_xlabel('Number of requests')
    ax.set_ylabel('Probability')
    ax.set_title(title)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('q3_poisson.png', dpi=150, bbox_inches='tight')
plt.show()

## Precomputed Transition Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, T, lbl in [(axes[0], _TRANS_1, 'Location 1'), (axes[1], _TRANS_2, 'Location 2')]:
    im = ax.imshow(T, cmap='Blues', aspect='auto')
    plt.colorbar(im, ax=ax, label='Transition probability')
    ax.set_xlabel('Cars at end of day (n_next)')
    ax.set_ylabel('Cars at start of day (n)')
    ax.set_title(f'Transition Matrix – {lbl}')

plt.tight_layout()
plt.savefig('q3_transitions.png', dpi=150, bbox_inches='tight')
plt.show()

print('Expected reward at loc1:')
print(f'  n=0: ${_EXP_REW_1[0]:.2f}, n=5: ${_EXP_REW_1[5]:.2f}, n=10: ${_EXP_REW_1[10]:.2f}, n=20: ${_EXP_REW_1[20]:.2f}')

## Policy Iteration

Starting from π₀(s) = 0 (never move), we alternate:
1. **Policy evaluation:** compute V^πₖ
2. **Policy improvement:** π_{k+1}(s) = argmax_a Q(s,a)

In [ ]:
V, policy = policy_iteration(theta=1e-2)
print('\nPolicy iteration complete!')
print(f'Final policy range: [{policy.min()}, {policy.max()}]')

## Results: Policy Heatmap and Value Function

In [ ]:
plot_results(V, policy, save_path='q3_jacks_policy.png')

## Interpretation

The optimal policy (matching Silver Lecture 3, p.15):
- When **loc1 has many cars** and **loc2 has few**: move cars from loc1→loc2 (negative action)
- When **loc2 has many cars** and **loc1 has few**: move cars from loc2→loc1 (positive action)
- The policy is not perfectly symmetric because rental rates differ (λ₂=4 > λ₁=3)